# Juman++ と KNP で係り受けと格関係を解析する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_parsing/knp.ipynb)

NLP クックブック 19-06 ／ 構文解析

京大の構文・格解析システム KNP で、文節・基本句の係り受けに加えて、述語ごとの格関係（誰が・何を・誰に）を得たい。KNP は形態素解析器 Juman++ の出力を入力にとるので、両方をソースからビルドする。

In [1]:
# Colab で 10〜20 分ほどかかる (KNP 辞書 2.8GB のダウンロードと展開、Juman++ と KNP のビルド。ダウンロード速度で変わる)
!apt-get install -y -q libtool > /dev/null
!wget -q https://github.com/ku-nlp/jumanpp/releases/download/v2.0.0-rc4/jumanpp-2.0.0-rc4.tar.xz
!tar xf jumanpp-2.0.0-rc4.tar.xz && mkdir -p jumanpp-2.0.0-rc4/bld
# テストなどを省き、jumanpp 本体だけをビルドしてインストールする
!cd jumanpp-2.0.0-rc4/bld && cmake .. -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1 && make -j2 jumanpp_v2 > /dev/null 2>&1 && cmake --install . > /dev/null
!git clone -q --depth 1 https://github.com/ku-nlp/knp
!wget -q http://lotus.kuee.kyoto-u.ac.jp/nl-resource/knp/dict/latest/knp-dict-latest-bin.zip
!unzip -q knp-dict-latest-bin.zip && cp -ars `pwd`/dict-bin/* knp/dict
!cd knp && ./autogen.sh > /dev/null 2>&1 && ./configure > /dev/null && make -j2 > /dev/null 2>&1 && make install > /dev/null 2>&1
# 辞書はインストール先にコピーされるので、ダウンロードと展開したもの (約 10GB) は消してよい
!rm -rf knp-dict-latest-bin.zip dict-bin jumanpp-2.0.0-rc4.tar.xz
!pip install -q rhoknp kwja
!jumanpp -v && du -sh /usr/local/share/knp/dict
!echo "太郎は花子に本を渡した。" | jumanpp | knp  # 既定は木構造の表示

In [2]:
from rhoknp import KNP, Jumanpp

# executable="/path/to/jumanpp" のようにバイナリの場所も指定できる。
# rhoknp は起動確認で空行を送るが、Juman++ rc4 は先頭が空行だと異常終了することがあり、そのとき警告が出るので、確認は省く
jumanpp = Jumanpp(skip_sanity_check=True)
knp = KNP(jumanpp=jumanpp)      # KNP(executable=...) も同様。既定のオプションは ["-tab"] (KNP 形式で出力)
sent = knp.apply_to_sentence("太郎と花子は、昨日京都で買った抹茶とお菓子を先生に渡した。")

DEP = {"D": "係り受け", "P": "並列", "A": "同格", "I": "部分並列"}
print("文節:")
for p in sent.phrases:
    print(f"  {p.text:　<8} → {p.parent.text if p.parent else '(文末)':　<8} {DEP[p.dep_type.value]}")
print([(m.text, m.reading, m.lemma, m.pos, m.subpos) for m in sent.morphemes][:6])

文節:
  太郎と　　　　　 → 花子は、　　　　 並列
  花子は、　　　　 → 渡した。　　　　 係り受け
  昨日　　　　　　 → 買った　　　　　 係り受け
  京都で　　　　　 → 買った　　　　　 係り受け
  買った　　　　　 → お菓子を　　　　 係り受け
  抹茶と　　　　　 → お菓子を　　　　 並列
  お菓子を　　　　 → 渡した。　　　　 係り受け
  先生に　　　　　 → 渡した。　　　　 係り受け
  渡した。　　　　 → (文末)　　　　 係り受け
[('太郎', 'たろう', '太郎', '名詞', '人名'), ('と', 'と', 'と', '助詞', '格助詞'), ('花子', 'はなこ', '花子', '名詞', '人名'), ('は', 'は', 'は', '助詞', '副助詞'), ('、', '、', '、', '特殊', '読点'), ('昨日', 'きのう', '昨日', '名詞', '時相名詞')]


In [3]:
# 格解析の結果: 述語の基本句ごとに、格フレームの ID と、各格に割り当てられた項
ARG = {"C": "表層の形から格が決まる (格助詞・時間表現など)", "N": "格が表層に現れない (提題・連体修飾など)",
       "O": "省略", "E": "外界"}

def arg_text(a):
    """項の表記 (文中の基本句なら その表記、「不特定:人」のような外界の項なら その名前)"""
    return a.base_phrase.text if hasattr(a, "base_phrase") else str(a)

for pas in sent.pas_list:
    print(f"{pas.predicate.text}  (格フレーム {pas.predicate.cfid})")
    print("  素性:", pas.predicate.base_phrase.features["格解析結果"])
    for case, args in pas.get_all_arguments(relax=False).items():
        for a in args:
            print(f"  {case}格: {arg_text(a):　<6} {a.type.value}: {ARG.get(a.type.value, '')}")

# 基本句の素性 (feature)。KNP が付けた情報は辞書として取り出せる
for bp in sent.base_phrases:
    keys = [k for k in ("用言", "体言", "係", "解析格", "時制", "SM-人") if k in bp.features]
    print(f"  {bp.text:　<6}", {k: bp.features[k] for k in keys})

import time
t = time.time()
for _ in range(20):
    knp.apply_to_sentence(sent.text)
print(f"1 文あたり {(time.time() - t) / 20 * 1000:.0f} ミリ秒 (Juman++ と KNP の合計)")

買った  (格フレーム 買う/かう:動1)
  素性: 買う/かう:動1:ガ/U/-/-/-/-;ヲ/N/抹茶/5/0/2;ヲ/N/菓子/6/0/2;ニ/U/-/-/-/-;ト/U/-/-/-/-;デ/C/京都/3/0/2;時間/C/昨日/2/0/2
  ヲ格: 抹茶と　　　 N: 格が表層に現れない (提題・連体修飾など)
  ヲ格: お菓子を　　 N: 格が表層に現れない (提題・連体修飾など)
  デ格: 京都で　　　 C: 表層の形から格が決まる (格助詞・時間表現など)
  時間格: 昨日　　　　 C: 表層の形から格が決まる (格助詞・時間表現など)
渡した。  (格フレーム 渡す/わたす:動1)
  素性: 渡す/わたす:動1:ガ/N/太郎/0/0/2;ガ/N/花子/1/0/2;ヲ/C/抹茶/5/0/2;ヲ/C/菓子/6/0/2;ニ/C/先生/7/0/2;ト/U/-/-/-/-;デ/U/-/-/-/-;カラ/U/-/-/-/-;ヘ/U/-/-/-/-;時間/U/-/-/-/-
  ガ格: 太郎と　　　 N: 格が表層に現れない (提題・連体修飾など)
  ガ格: 花子は、　　 N: 格が表層に現れない (提題・連体修飾など)
  ヲ格: 抹茶と　　　 C: 表層の形から格が決まる (格助詞・時間表現など)
  ヲ格: お菓子を　　 C: 表層の形から格が決まる (格助詞・時間表現など)
  ニ格: 先生に　　　 C: 表層の形から格が決まる (格助詞・時間表現など)
  太郎と　　　 {'体言': True, '係': 'ト格', 'SM-人': True}
  花子は、　　 {'体言': True, '係': '未格', '解析格': 'ガ', 'SM-人': True}
  昨日　　　　 {'体言': True, '係': '無格', '解析格': '時間'}
  京都で　　　 {'体言': True, '係': 'デ格', '解析格': 'デ'}
  買った　　　 {'用言': '動', '係': '連格', '時制': '過去'}
  抹茶と　　　 {'体言': True, '係': 'ト格'}
  お菓子を　　 {'体言': True, '係': 'ヲ格', '解析格': 'ヲ'}
  先生に　　　 {'体言': True, '係': 'ニ格

In [4]:
# 同じ文を KWJA (前のレシピ) で解析して比べる。出力はどちらも KNP 形式なので rhoknp で同じように扱える
import subprocess
from rhoknp import Document

out = subprocess.run(["kwja", "--text", sent.text, "--tasks", "char,word", "--model-size", "base"],
                     capture_output=True, text=True, check=True).stdout
kw = Document.from_knp(out).sentences[0]

# 文節の区切りが両者で同じであることを前提に、文節を順に並べて比べる
assert [p.text for p in sent.phrases] == [q.text for q in kw.phrases]
print(f"{'文節':　<8} {'KNP':　<10} {'KWJA':　<10}")
for p, q in zip(sent.phrases, kw.phrases):
    a = (p.parent.text if p.parent else "(文末)") + p.dep_type.value
    b = (q.parent.text if q.parent else "(文末)") + q.dep_type.value
    print(f"{p.text:　<8} {a:　<10} {b:　<10}", "" if a == b else "← 違う")

def args_of(pas):
    return {case: [arg_text(a) for a in v] for case, v in pas.get_all_arguments(relax=False).items()}

# 述語は基本句の番号で対応づける (基本句の区切りも同じであることを確かめておく)
assert [bp.text for bp in sent.base_phrases] == [bp.text for bp in kw.base_phrases]
knp_pas = {pas.predicate.base_phrase.index: pas for pas in sent.pas_list}
kw_pas = {pas.predicate.base_phrase.index: pas for pas in kw.pas_list}
for i in sorted(knp_pas.keys() | kw_pas.keys()):
    print(sent.base_phrases[i].text)
    print("  KNP :", args_of(knp_pas[i]) if i in knp_pas else "-")
    print("  KWJA:", args_of(kw_pas[i]) if i in kw_pas else "-")

文節　　　　　　 KNP　　　　　　　 KWJA　　　　　　
太郎と　　　　　 花子は、P　　　　　 花子は、P　　　　　 
花子は、　　　　 渡した。D　　　　　 渡した。D　　　　　 
昨日　　　　　　 買ったD　　　　　　 買ったD　　　　　　 
京都で　　　　　 買ったD　　　　　　 買ったD　　　　　　 
買った　　　　　 お菓子をD　　　　　 お菓子をD　　　　　 
抹茶と　　　　　 お菓子をP　　　　　 お菓子をP　　　　　 
お菓子を　　　　 渡した。D　　　　　 渡した。D　　　　　 
先生に　　　　　 渡した。D　　　　　 渡した。D　　　　　 
渡した。　　　　 (文末)D　　　　　 (文末)D　　　　　 
買った
  KNP : {'ヲ': ['抹茶と', 'お菓子を'], 'デ': ['京都で'], '時間': ['昨日']}
  KWJA: {'ガ': ['花子は、'], 'ヲ': ['お菓子を'], 'デ': ['京都で'], '時間': ['昨日']}
渡した。
  KNP : {'ガ': ['太郎と', '花子は、'], 'ヲ': ['抹茶と', 'お菓子を'], 'ニ': ['先生に']}
  KWJA: {'ガ': ['太郎と'], 'ヲ': ['抹茶と'], 'ニ': ['先生に']}


- KNP は、Juman++（または JUMAN）の形態素解析結果を入力にとり、文節・基本句の係り受け、格関係、照応関係を出力する。人手で書いたルール（`rule/` の下）で形態素・文節に素性を付けて係り受けの候補を作り、並列構造は前後の文節列の類似度で検出する。候補は、ウェブテキストから自動構築した大規模な格フレームに基づく確率で評価する（README とマニュアル `doc/manual.pdf` による）。マニュアルによると、出力するのは最もよい 1 つの解だけである。
- この文では、文節の係り先と種類 (D/P) は KNP と KWJA で全く同じになった。「太郎と → 花子は」「抹茶と → お菓子を」が並列 (P)、「京都で」「昨日」は「買った」に係る。
- 格解析の違い: KNP は並列の両方（「渡した」のガ格に「太郎と」「花子は、」、ヲ格に「抹茶と」「お菓子を」）を項に入れるが、KWJA は並列の一方だけを項にした（「渡した」のガ格は「太郎と」、ヲ格は「抹茶と」、「買った」のヲ格は「お菓子を」）。また「買った」のガ格は、ガ格になる文節が「買った」に係っていないため KNP では空き（`格解析結果` の `ガ/U/-`）だが、KWJA は係り受けのない「花子は、」をガ格として出した。KNP の照応解析 (`-anaphora`) は試していない。
- `格解析結果` の各項は `格/種類/表記/基本句番号/...` の形。種類は、C が「述語に係る格要素で、表層の形から格が決まるもの」（格助詞の付いた「先生に」のほか、時間格の「昨日」も C）、N が「格が表層に現れないもの」（提題の「花子は、」や、連体修飾で係る「お菓子を」（「買った」のヲ格）など）、U が「割り当てなし」。並列の前の要素（「太郎と」「抹茶と」）は後ろの要素と同じ種類になっている。rhoknp は U 以外を `BasePhrase.pas` (`Sentence.pas_list`) に読み込む。格フレームの ID (`渡す/わたす:動1`) は、1 つの述語に用法ごとに複数ある格フレームのどれを選んだかを表す。
- 基本句の素性には、係りの形（`係:未格` など）、格解析で決まった格 (`解析格:ガ`)、意味素性 (`SM-人`)、時制などが入る。素性の一覧は KNP の `doc/knp_feature.pdf` にある。
- 出力は KWJA と同じ KNP 形式で、rhoknp で同じように読める。形態素は JUMAN の品詞体系（「名詞/人名」「助詞/副助詞」「特殊/読点」など）。コマンドラインの `knp` は既定で木構造を表示し、rhoknp は `-tab`（表形式）で呼ぶ。
- 速さ: rhoknp は Juman++ と KNP のプロセスを起動したまま使い回すので、この文では 1 文あたり 0.15〜0.2 秒ほどだった（上の計測。CPU 2 コアで 2 回計って 152 ミリ秒と 184 ミリ秒）。KWJA はコマンドを呼ぶたびにモデルを読み込み、GPU なし（CPU 2 コア）では初回（モデルのダウンロードを含む）に 50 秒ほどかかった。
- `pip install kwja` で、ほかのパッケージとの依存関係の衝突の警告 (ERROR: pip's dependency resolver ...) が出るが、ここでは無視してよい。KWJA はコマンドとして別プロセスで動かしているので、ノートブックの再起動も要らない（KWJA を Python から import して使う場合は、前のレシピの注意のとおり再起動が必要になることがある）。
- ビルドの手間: KNP は GitHub にリリースがなく、git から autogen.sh（libtool が必要）でビルドする。辞書（格フレームなど）は別配布の 2.8GB の zip で、インストール先に 7GB ほどを使う。Juman++ rc4 は、先頭の入力が空行だと異常終了することがある（毎回ではない）。rhoknp の起動確認は空行を送るので、そのとき「failed to start Juman++」の警告が出る。これを避けるため `skip_sanity_check=True` を付ける。